# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`
This notebook provides an example workflow to load and explore the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # Keeps access object, do not subscript
print(f"Title: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list the record sets and their fields, referencing all by their `@id`.

In [ ]:
print("All record sets in this dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}, description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. **All Croissant entities in the following sections are referenced by their `@id` fields.**

We'll extract data from each record set.

In [ ]:
# Get all record set @id's
record_sets_ids = [r.id for r in dataset.metadata.record_sets]

dataframes = {}
for rset_id in record_sets_ids:
    # Load records, with Croissant referencing by @id
    records = list(dataset.records(record_set=rset_id))
    dataframes[rset_id] = pd.DataFrame(records)

for rset_id, df in dataframes.items():
    print(f"\nRecord set: {rset_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
We'll perform exploratory analysis on one of the record sets. _Refer to fields by their Croissant `@id`._

Typical steps include:
- Filtering records,
- Normalizing a numeric field,
- Grouping by a categorical field.
**Edit the field and group names below to match the actual dataset fields for your analysis.**

In [ ]:
# Choose one of the record sets by its @id:
record_set_id = record_sets_ids[0]  # or choose another by index
df = dataframes[record_set_id]

# List fields to pick a numeric field for demonstration
print("Available columns for EDA:")
print(df.columns.tolist())

# Select a numeric field (by @id) that exists in the first record set
try:
    numeric_field = next(col for col in df.columns if df[col].dtype in [int, float])
except StopIteration:
    numeric_field = df.columns[0]  # fallback

print(f"Using numeric field '@id': {numeric_field}")

threshold = df[numeric_field].mean()  # Example threshold: the mean
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a non-numeric field if one exists
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the DTI dataset via its Croissant schema, inspected the schema's record sets and field `@id`s, loaded data, performed data cleaning and analysis, and visualized the distributions of selected fields.

- **All referencing of fields and record sets was done by their Croissant `@id`.**
- The workflow can be extended to feature engineering, model training, or advanced visualizations as needed.